In [ ]:
"""
%pip install qiskit-ibm-runtime==0.47.0
%pip install samplomatic==0.18.0
%pip install qiskit-machine-learning==0.9.0
%pip install qiskit-nature==0.7.2
%pip install qiskit-aer==0.17.2
%pip install pylatexenc==2.10
%pip install qiskit==2.3.0
%pip install numpy==2.4.2
%pip install pyscf==2.12.1
"""

'\n%pip install qiskit-ibm-runtime==0.47.0\n%pip install samplomatic==0.18.0\n%pip install qiskit-machine-learning==0.9.0\n%pip install qiskit-nature==0.7.2\n%pip install qiskit-aer==0.17.2\n%pip install pylatexenc==2.10\n%pip install qiskit==2.3.0\n%pip install numpy==2.4.2\n%pip install pyscf==2.12.1\n'

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
from qiskit_aer import AerSimulator
from joblib import Parallel, delayed
from qiskit_ibm_runtime import EstimatorV2
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_algorithms import MinimumEigensolverResult
from qiskit_ibm_runtime.fake_provider import FakeBoston
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_machine_learning.optimizers import GradientDescent
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


# 1°: Building the Molecular Problem


In [ ]:
# Creating the molecular geometry of BeH2 with STO-3G as minimal basis set and
# H-Be distance of 1.326 angstrom and 180 degrees,

driver = PySCFDriver(
        atom="""H -1.326, 0.0, 0.0
                Be 0.0, 0.0, 0.0
                H 1.326, 0.0, 0.0
             """,
        basis='sto3g',
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM
)

molecule_problem = driver.run()

In [ ]:
# The original space has approximately 6 electrons, 6 space orbitals, and 12
# spin orbitals.
# Here we limit the active space to only 4 electrons and 3 space orbitals and
# work with the reduced molecule_problem

active_space_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)

reduced_molecule_problem = active_space_transformer.transform(molecule_problem)

# 2°: The Hamiltonian in Terms of Qubits

In [ ]:
# Here we calculate the Hamiltonian of the second quantization after reduction
# with CAS.
second_q_hamiltonian = reduced_molecule_problem.second_q_ops()[0]

In [ ]:
# Here we use the Jordan Wigner mapping
jordan_wigner_mapper = JordanWignerMapper()

qubit_op = jordan_wigner_mapper.map(second_q_hamiltonian)

In [ ]:
# Finally, we obtain the number of qubits and operator in terms of Pauli matrices.
num_qubits = qubit_op.num_qubits
print(f"Number of qubits = {num_qubits}")
print(f"Hamiltonian: {qubit_op}")

Number of qubits = 6
Hamiltonian: SparsePauliOp(['IIIIII', 'IIIIIZ', 'IIIIZI', 'IIIZII', 'IIZIII', 'IZIIII', 'ZIIIII', 'IIIIZZ', 'IIIZIZ', 'IIZIIZ', 'IZIIIZ', 'ZIIIIZ', 'IYYIYY', 'IXXIYY', 'IYYIXX', 'IXXIXX', 'YZYYZY', 'XZXYZY', 'YZYXZX', 'XZXXZX', 'IIIZZI', 'IIZIZI', 'IZIIZI', 'ZIIIZI', 'YYIYYI', 'XXIYYI', 'YYIXXI', 'XXIXXI', 'IIZZII', 'IZIZII', 'ZIIZII', 'IZZIII', 'ZIZIII', 'ZZIIII'],
              coeffs=[-2.86740153+0.j,  0.32161792+0.j,  0.31034865+0.j,  0.12889394+0.j,
  0.32161792+0.j,  0.31034865+0.j,  0.12889394+0.j,  0.06196671+0.j,
  0.08001574+0.j,  0.09975793+0.j,  0.10309655+0.j,  0.09238729+0.j,
  0.04112984+0.j,  0.04112984+0.j,  0.04112984+0.j,  0.04112984+0.j,
  0.01237154+0.j,  0.01237154+0.j,  0.01237154+0.j,  0.01237154+0.j,
  0.08557179+0.j,  0.10309655+0.j,  0.10888574+0.j,  0.08924797+0.j,
  0.00367617+0.j,  0.00367617+0.j,  0.00367617+0.j,  0.00367617+0.j,
  0.09238729+0.j,  0.08924797+0.j,  0.11246476+0.j,  0.06196671+0.j,
  0.08001574+0.j,  0.08557179+0.j])


# 3°: Ansatz Circuit Construction


In [ ]:
# Here we construct the HF state within the CAS and JW mapping.
hf_initial_state = HartreeFock(
      num_particles=reduced_molecule_problem.num_particles,
      num_spatial_orbitals=reduced_molecule_problem.num_spatial_orbitals,
      qubit_mapper=jordan_wigner_mapper
)

In [ ]:
# Here we build the ansatz
ansatz = UCCSD(
          reduced_molecule_problem.num_spatial_orbitals,
          reduced_molecule_problem.num_particles,
          initial_state=hf_initial_state,
          qubit_mapper=jordan_wigner_mapper
)

# 4°: Transpilation and Simulator Settings

In [ ]:
# This small function was created to generate the ansatz and observables in
# terms of Instruction Set Architecture (ISA) operators.
def transpile_to_isa(backend, initial_layout, seed=14):
  target = backend.target

  pm = generate_preset_pass_manager(
       target=target,
       initial_layout=initial_layout,
       optimization_level=3,
       seed_transpiler=seed,
       layout_method='sabre',
       routing_method='sabre'
  )

  ansatz_isa = pm.run(ansatz)
  isa_observables = qubit_op.apply_layout(ansatz_isa.layout)
  return ansatz_isa, isa_observables

# 5°: Measuring Eigenvalues and Energies

In [ ]:
# Here we add the energy of active space to the frozen energies of the core
# plus nuclear repulsion.
def interpret_exp_val(exp_val, problem):
    sol = MinimumEigensolverResult()
    sol.eigenvalue = np.real(exp_val)
    return problem.interpret(sol).total_energies[0]

# 6°:Configuring Parallel Execution Function

In [ ]:
def parallel_optimization(
    run_idx: int,
    seed_sim: int,
    shots: int,
    maxiter: int,
    initial_layout:list
):

  # Initialization of the Aer simulator with FakeProvider
  backend = AerSimulator.from_backend(FakeBoston())
  backend.options.seed_simulator = seed_sim

  # Ansatz transpilation and observables for the ISA
  ansatz_isa, isa_observables = transpile_to_isa(
                                backend,
                                initial_layout,
                                )

  # Declaring and configuring the Estimator for calculating expected values.
  estimator = EstimatorV2(mode=backend)
  estimator.options.default_shots = shots

  # The energies and parameters per iteration will be stored here.
  energy_data = []
  params_data = []

  # Initializing parameters as all at 0. This means we will start in the HF state.
  initial_params = np.zeros(ansatz_isa.num_parameters)
  bounds = ([(-np.pi, np.pi)] * ansatz_isa.num_parameters)

  # Callback function declaration
  def optimizer_callback(ne, params, value, step):
    energy_data.append(value)
    params_data.append(params.tolist())

  # Statement of the cost function
  def energy_cost_function(params):
    estimator_job = estimator.run([(ansatz_isa, isa_observables, params)])
    estimator_exp_val = estimator_job.result()[0].data.evs
    return float(estimator_exp_val)

  # Optimizer instance with settings
  optimizer = GradientDescent(
              maxiter=maxiter,
              learning_rate=0.01,
              tol=1e-8,
              callback=optimizer_callback
            )

  # The optimization process takes place here.
  t0 = time.perf_counter()
  result = optimizer.minimize(fun=energy_cost_function, x0=initial_params, bounds=bounds)
  t1 = time.perf_counter()

  return {
        "optimizer": "GD",
        "run": run_idx,
        "seed_sim": seed_sim,
        "steps_requested": optimizer.settings.get('maxiter'),
        "steps_done": result.nit,
        "cost_function_evaluation": result.nfev,
        "energies_trajectory_len": len(energy_data),
        "execution_time": t1 - t0,
        "final_energy": float(result.fun),
        "optimal_params": result.x.tolist(),
        "energies_trajectory": energy_data,
        "params_trajectory": params_data,
  }

# 7°: Running VQE in parallel on CPU cores

In [ ]:
maxiter = 1000
shots = 10_000
runs = 50
seed_sim = 42
initial_layout= [43, 42, 56, 63, 62, 61]

In [ ]:
n_jobs = os.cpu_count()
print(f"Running {runs} GD runs in parallel (CPU) | n_jobs={n_jobs} | shots={shots} | iteration={maxiter}")

Running 50 GD runs in parallel (CPU) | n_jobs=44 | shots=10000 | iteration=1000


In [15]:
results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
    delayed(parallel_optimization)(
        run_idx=i,
        seed_sim=seed_sim+i,
        shots=shots,
        maxiter=maxiter,
        initial_layout=initial_layout
    )
    for i in range(runs)
)

[Parallel(n_jobs=44)]: Using backend LokyBackend with 44 concurrent workers.
[Parallel(n_jobs=44)]: Done   5 out of  50 | elapsed: 801.1min remaining: 7209.8min
[Parallel(n_jobs=44)]: Done  11 out of  50 | elapsed: 803.0min remaining: 2847.1min
[Parallel(n_jobs=44)]: Done  17 out of  50 | elapsed: 804.1min remaining: 1560.9min
[Parallel(n_jobs=44)]: Done  23 out of  50 | elapsed: 804.7min remaining: 944.7min
[Parallel(n_jobs=44)]: Done  29 out of  50 | elapsed: 805.3min remaining: 583.1min
[Parallel(n_jobs=44)]: Done  35 out of  50 | elapsed: 806.2min remaining: 345.5min
[Parallel(n_jobs=44)]: Done  41 out of  50 | elapsed: 806.7min remaining: 177.1min
[Parallel(n_jobs=44)]: Done  47 out of  50 | elapsed: 1304.9min remaining: 83.3min
[Parallel(n_jobs=44)]: Done  50 out of  50 | elapsed: 1305.9min finished


In [16]:
results.sort(key=lambda d: d["run"])

# 8°: Adding the constant energies

In [17]:
for run_data in results:

    raw_final_energy = run_data["final_energy"]
    true_final_energy = float(interpret_exp_val(raw_final_energy, reduced_molecule_problem))

    run_data["final_energy"] = true_final_energy

    raw_trajectory = run_data["energies_trajectory"]

    true_trajectory = [
        interpret_exp_val(step_energy, reduced_molecule_problem)
        for step_energy in raw_trajectory
    ]

    run_data["energies_trajectory"] = true_trajectory

# 9º Save the data for later analysis.

In [18]:
df = pd.DataFrame(results)

df["optimal_params_json"] = df["optimal_params"].apply(json.dumps)
df["energies_trajectory_json"] = df["energies_trajectory"].apply(json.dumps)
df["params_trajectory_json"] = df["params_trajectory"].apply(json.dumps)

df.to_csv(
    "BEH2_VQE_GD_HARDWARE_NOISE.csv",
    columns=[
        "optimizer",
        "run",
        "seed_sim",
        "steps_requested",
        "steps_done",
        "cost_function_evaluation",
        "energies_trajectory_len",
        "execution_time",
        "final_energy",
        "optimal_params_json",
        "energies_trajectory_json",
        "params_trajectory_json",
    ],
    index=False,
)

print("\nSaved: BEH2_VQE_GD_HARDWARE_NOISE.csv")
print("Done.")


Saved: BEH2_VQE_GD_HARDWARE_NOISE.csv
Done.
